# Домашняя работа
Мацюк Константин Владимирович  
НПИбд-02-25  
1032253640

## Класс df

In [ ]:
from sage.all import *
x, y, t, v, C = var('x y t v C', domain='real')

def iszero(e):
    return bool(SR(e).full_simplify() == 0)

def primitive(e, z):
    """Первообразная; если CAS не вычисляет её, сохраняем квадратуру."""
    e = SR(e).full_simplify()
    try:
        return SR(integral(e, z)).full_simplify()
    except (ValueError, RuntimeError, NotImplementedError):
        return integral(e, z, hold=True)

class df:
    def __init__(self, p, q):
        self.p, self.q = SR(p), SR(q)
        if iszero(self.p) and iszero(self.q):
            raise ValueError('Нулевая форма не задаёт ОДУ.')

    def list(self):
        return [self.p, self.q]

    def _rhs(self):
        if iszero(self.q):
            raise ValueError("q = 0: нельзя представить уравнение как производную y по x.")
        return SR(-self.p/self.q).full_simplify()

    def is_exact(self):
        return iszero(diff(self.p, y) - diff(self.q, x))

    def potential(self, method=2):
        if not self.is_exact():
            raise ValueError('Форма не точна: p_y != q_x.')
        if method in (1, 'radial'):
            origin = {x: 0, y: 0}
            if any(iszero(a.denominator().subs(origin)) for a in self.list()):
                raise ValueError('Форма не определена в начале координат.')
            a = SR(self.p.subs({x: t*x, y: t*y})*x
                     + self.q.subs({x: t*x, y: t*y})*y).full_simplify()
            return SR(integral(a, t, 0, 1)).full_simplify()
        if method not in (2, 'successive'):
            raise ValueError('Неизвестный способ.')
        u = primitive(self.p, x)
        rest = SR(self.q - diff(u, y)).full_simplify()
        if not iszero(diff(rest, x)):
            raise NotImplementedError('CAS не подтвердил независимость остатка от x.')
        ans = SR(u + primitive(rest, y)).full_simplify()
        if not (iszero(diff(ans, x)-self.p) and iszero(diff(ans, y)-self.q)):
            raise NotImplementedError('CAS не подтвердил производные первообразной.')
        return ans

    def integral(self, contour, a=0, b=1):
        X, Y = map(SR, contour)
        value = self.p.subs({x: X, y: Y})*diff(X, t)
        value += self.q.subs({x: X, y: Y})*diff(Y, t)
        return SR(integral(value, t, a, b)).full_simplify()

    def symmetry_residual(self, xi, eta):
        h = self._rhs()
        xi, eta = SR(xi), SR(eta)
        return SR(diff(eta,x) + (diff(eta,y)-diff(xi,x))*h
                    - diff(xi,y)*h**2 - xi*diff(h,x) - eta*diff(h,y)).full_simplify()

    def integrate_symmetry(self, xi, eta):
        xi, eta = SR(xi), SR(eta)
        if not iszero(self.symmetry_residual(xi, eta)):
            raise ValueError('Предложенная пара не подтверждена как симметрия.')
        denominator = SR(self.p*xi + self.q*eta).full_simplify()
        if iszero(denominator):
            raise ValueError('Касательная симметрия: формула множителя неприменима.')
        mu = SR(1/denominator).full_simplify()
        exact_form = df(SR(mu*self.p).full_simplify(), SR(mu*self.q).full_simplify())
        if not exact_form.is_exact():
            raise ArithmeticError('Не подтверждена точность формы.')
        I = exact_form.potential(method=2)
        return {'xi': xi, 'eta': eta, 'mu': mu, 'I': I}

    def integral_residual(self, I):
        # Проверка без деления на q: q I_x - p I_y = 0.
        return SR(self.q*diff(I,x) - self.p*diff(I,y)).full_simplify()


def symmetry_fx(self):
    h = self._rhs()
    A = SR(-diff(h, x)/h).full_simplify()
    if not bool(SR(diff(A, y)).full_simplify() == 0):
        return None
    xi = SR(exp(integral(A, x))).full_simplify()
    return (xi, SR(0)) if bool(self.symmetry_residual(xi, 0) == 0) else None

def symmetry_fy(self):
    h = self._rhs()
    B = SR(-diff(h, x)/h**2).full_simplify()
    if not bool(SR(diff(B, x)).full_simplify() == 0):
        return None
    xi = SR(exp(integral(B, y))).full_simplify()
    return (xi, SR(0)) if bool(self.symmetry_residual(xi, 0) == 0) else None

def symmetry_eta_fx(self):
    h = self._rhs()
    A = SR(diff(h, y)).full_simplify()
    if not bool(SR(diff(A, y)).full_simplify() == 0):
        return None
    eta = SR(exp(integral(A, x))).full_simplify()
    return (SR(0), eta) if bool(self.symmetry_residual(0, eta) == 0) else None

def symmetry_eta_fy(self):
    h = self._rhs()
    B = SR(diff(h, y)/h).full_simplify()
    if not bool(SR(diff(B, x)).full_simplify() == 0):
        return None
    eta = SR(exp(integral(B, y))).full_simplify()
    return (SR(0), eta) if bool(self.symmetry_residual(0, eta) == 0) else None

def show_result(r):
    for key in ('xi', 'eta', 'mu', 'I'):
        if key in r:
            print(key, '=', r[key])

## 1. Однородные уравнения
Добавьте к методам класса `df` процедуру, которая определяет однородные дифференциальные уравнения и интегрирует их. Проверьте её работу на
$$
(x^2+y^2)\,dx+xy\,dy=0.
$$

In [ ]:
def is_homogeneous(self):
    h = self._rhs()
    return iszero(x*diff(h,x) + y*diff(h,y))

def solve_homogeneous(self):
    if not self.is_homogeneous():
        return None
    D = SR(x*self.p + y*self.q).full_simplify()
    if iszero(D):
        I = y/x
        return {'xi': x, 'eta': y, 'mu': SR(1/(x*self.q)).full_simplify(),
                'I': I, 'case': 'h = y/x; интеграл получен напрямую'}
    return self.integrate_symmetry(x, y)

df.is_homogeneous = is_homogeneous
df.solve_homogeneous = solve_homogeneous

ode1 = df(x**2 + y**2, x*y)
print('Однородное:', ode1.is_homogeneous())
r1 = ode1.solve_homogeneous()
show_result(r1)
print('Проверка найденного интеграла (ожидается 0):',
      ode1.integral_residual(r1['I']))

J1 = x**4 + 2*x**2*y**2
print('Полиномиальный первый интеграл:', J1, '= C')
print('Проверка полиномиального интеграла (ожидается 0):',
      ode1.integral_residual(J1))

**Ответ:** уравнение однородное,
$$
x^4+2x^2y^2=C.
$$
Как дифференциальная форма уравнение также допускает вертикальную интегральную кривую $x=0$.

## 2. Симметрия $\xi=f(x)$, $\eta=0$
Опишите ОДУ, имеющие симметрии типа $\xi=f(x)$, $\eta=0$.

**Ответ:** уравнения с разделяющимися переменными:
$$
y'=\frac{A(y)}{f(x)}.
$$
При $f(x)A(y)\ne0$ общий интеграл:
$$
\int\frac{dy}{A(y)}-\int\frac{dx}{f(x)}=C.
$$
Здесь и далее $A$ — произвольная функция. Нули $A(y)$ дают постоянные решения.

In [ ]:
eq2 = df(-x*y, 1)
sym2 = eq2.symmetry_fx()
print('Симметрия:', sym2)
if sym2 is not None:
    r2 = eq2.integrate_symmetry(*sym2)
    print('Множитель:', r2['mu'])
    print('Первый интеграл:', r2['I'], '= C')
    print('Проверка точности:', SR(diff(r2['mu']*eq2.p,y) - diff(r2['mu']*eq2.q,x)).full_simplify())


## 3. Симметрия $\xi=f(y)$, $\eta=0$
Опишите ОДУ, имеющие симметрии типа $\xi=f(y)$, $\eta=0$.

**Ответ:** уравнения вида
$$
y'=\frac{1}{x\dfrac{f'(y)}{f(y)}+A(y)}.
$$
При $f(y)\ne0$ после перехода к $x=x(y)$ получаем линейное уравнение:
$$
\frac{dx}{dy}-\frac{f'(y)}{f(y)}x=A(y).
$$
Общий интеграл:
$$
\frac{x}{f(y)}-\int\frac{A(y)}{f(y)}\,dy=C.
$$
Отдельный случай $y'=0$ также допускает такую симметрию.

In [ ]:
eq3 = df(-1/(x*y + y**2 + 1), 1)
sym3 = eq3.symmetry_fy()
print('Симметрия:', sym3)
if sym3 is not None:
    r3 = eq3.integrate_symmetry(*sym3)
    print('Множитель:', r3['mu'])
    print('Первый интеграл:', r3['I'], '= C')
    print('Проверка точности:', SR(diff(r3['mu']*eq3.p,y) - diff(r3['mu']*eq3.q,x)).full_simplify())


## 4. Симметрия $\xi=0$, $\eta=f(x)$
Опишите ОДУ, имеющие симметрии типа $\xi=0$, $\eta=f(x)$.

**Ответ:** линейные ОДУ первого порядка:
$$
y'=\frac{f'(x)}{f(x)}y+A(x).
$$
При $f(x)\ne0$ общий интеграл:
$$
\frac{y}{f(x)}-\int\frac{A(x)}{f(x)}\,dx=C.
$$

In [ ]:
eq4 = df(-(x*y + x**2), 1)
sym4 = eq4.symmetry_eta_fx()
print('Симметрия:', sym4)
if sym4 is not None:
    r4 = eq4.integrate_symmetry(*sym4)
    print('Множитель:', r4['mu'])
    print('Первый интеграл:', r4['I'], '= C')
    print('Проверка точности:', SR(diff(r4['mu']*eq4.p,y) - diff(r4['mu']*eq4.q,x)).full_simplify())


## 5. Симметрия $\xi=0$, $\eta=f(y)$
Опишите ОДУ, имеющие симметрии типа $\xi=0$, $\eta=f(y)$.

**Ответ:** уравнения с разделяющимися переменными:
$$
y'=A(x)f(y).
$$
При $f(y)\ne0$ общий интеграл:
$$
\int\frac{dy}{f(y)}-\int A(x)\,dx=C.
$$
Нули $f(y)$ дают постоянные решения.

In [ ]:
eq5 = df(-x**2*exp(y), 1)
sym5 = eq5.symmetry_eta_fy()
print('Симметрия:', sym5)
if sym5 is not None:
    r5 = eq5.integrate_symmetry(*sym5)
    print('Множитель:', r5['mu'])
    print('Первый интеграл:', r5['I'], '= C')
    print('Проверка точности:', SR(diff(r5['mu']*eq5.p,y) - diff(r5['mu']*eq5.q,x)).full_simplify())


## 6. Симметрия $\xi=f(x)g(y)$, $\eta=0$
Добавьте к методам класса `df` процедуру, которая определяет ОДУ, имеющие симметрию вида $\xi=f(x)g(y)$, $\eta=0$, находит эту симметрию, а по ней множитель и интеграл заданного уравнения.

In [ ]:
def symmetry_fg(self):
    """Ищет xi=f(x)g(y), eta=0; None означает: критерии не подтверждены."""
    h = self._rhs()
    if iszero(h):
        # Любая xi подходит, но горизонтальная симметрия касательна.
        return {'f': SR(1), 'g': SR(1), 'xi': SR(1), 'eta': SR(0),
                'a': SR(0), 'b': SR(0), 'case': 'h = 0'}

    R = SR(diff(diff(h,x)/h, y)/h**2).full_simplify()
    if iszero(R):
        a = SR(-diff(h,x)/h).full_simplify()
        b = SR(0)
        case = 'R = 0: разделяющиеся переменные'
    else:
        a = SR(diff(R,x)/R).full_simplify()
        if not iszero(diff(a,y)):
            return None
        z = SR(1/h).full_simplify()
        b = SR(diff(z,x)-a*z).full_simplify()
        case = 'R != 0: общий случай'

    if not (iszero(diff(a,y)) and iszero(diff(b,x))):
        return None
    if not iszero(diff(h,x)+a*h+b*h**2):
        return None

    f = SR(exp(primitive(a,x))).full_simplify()
    g = SR(exp(primitive(b,y))).full_simplify()
    xi = SR(f*g).full_simplify()
    if not iszero(self.symmetry_residual(xi, 0)):
        raise NotImplementedError('CAS не подтвердил найденную симметрию.')
    return {'f': f, 'g': g, 'a': a, 'b': b, 'xi': xi,
            'eta': SR(0), 'R': R, 'case': case}

def solve_fg(self, quadrature=False):
    symmetry = self.symmetry_fg()
    if symmetry is None:
        return None
    if iszero(self.p):
        # y'=0 интегрируем напрямую. mu не получен из касательной xi.
        return dict(symmetry, mu=SR(1/self.q).full_simplify(), I=y,
                    note='y=C; mu=1/q найден напрямую, а не из 1/(p*xi).')
    if quadrature:
        # Построение интеграла из нормальной формы без попытки
        # вычислить неэлементарную квадратуру по y.
        f, g, b = symmetry['f'], symmetry['g'], symmetry['b']
        F = primitive(1/f, x)
        alpha = SR(1/(self._rhs()*f) - b*F).full_simplify()
        if not iszero(diff(alpha, x)):
            raise NotImplementedError('CAS не подтвердил alpha=alpha(y).')
        s22, y_base = var('s22 y_base', domain='real')
        integrand = SR(alpha/g).full_simplify().subs({y: s22})
        I = F/g - integral(integrand, s22, y_base, y, hold=True)
        mu = SR(1/(self.p*symmetry['xi'])).full_simplify()
        # Производные определённого интеграла по основной теореме анализа.
        Ix = SR(diff(F, x)/g).full_simplify()
        Iy = SR(-F*diff(g, y)/g**2-alpha/g).full_simplify()
        return dict(symmetry, F=F, alpha=alpha, mu=mu, I=I,
                    Ix=Ix, Iy=Iy, integrand=integrand,
                    integration_variable=s22, basepoint=y_base)
    result = self.integrate_symmetry(symmetry['xi'], 0)
    return dict(symmetry, **{k: result[k] for k in ('mu', 'I')})

df.symmetry_fg = symmetry_fg
df.solve_fg = solve_fg

Проверим метод на уравнении
$$
-y^2\,dx+(x+y)\,dy=0.
$$

In [ ]:
ode6 = df(-y**2, x+y)
r6 = ode6.solve_fg()
show_result(r6)

**Ответ:**
$$
\xi=e^{-1/y},\qquad \eta=0,\qquad
\mu=-\frac{e^{1/y}}{y^2}.
$$
$$
xe^{1/y}+\operatorname{Ei}(1/y)=C,\qquad y\ne0.
$$
Дополнительно $y=0$ — решение при $x\ne0$.

## 7. Уравнение Чеб-Терраба
Проверьте, что созданная вами программа может проинтегрировать уравнение (22) из статьи Чеб-Терраба.
$$
y'=-\frac{(y+b)^2}{(x+a)\left[1+(y+b)^2(x+a)\sin y\right]}.
$$

In [ ]:
aa, bb = var('a b', domain='real')
p22 = (y+bb)**2
q22 = (x+aa)*(1+(y+bb)**2*(x+aa)*sin(y))
ode22 = df(p22, q22)

r22 = ode22.solve_fg(quadrature=True)
show_result(r22)

print('Проверка первого интеграла (ожидается 0):',
      SR(q22*r22['Ix'] - p22*r22['Iy']).full_simplify())
print('Проверка решения y=-b (ожидается 0):',
      SR(ode22._rhs().subs({y: -bb})).full_simplify())

**Ответ:**
$$
\xi=(x+a)^2e^{-1/(y+b)},\qquad \eta=0,
$$
$$
\mu=\frac{e^{1/(y+b)}}{(x+a)^2(y+b)^2}.
$$
Общий интеграл:
$$
-\frac{e^{1/(y+b)}}{x+a}
+\int_{y_0}^{y}e^{1/(s+b)}\sin s\,ds=C.
$$
Результат совпадает с формулой (24) статьи. Дополнительное решение: $y=-b$ при $x+a\ne0$.

Область основного решения: $x+a\ne0$, $y+b\ne0$, $1+(y+b)^2(x+a)\sin y\ne0$; интервал интегрирования не содержит $-b$.